# Week 42, Day 5: Three-factor rules: eligibility traces and neuromodulators

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w042/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Finding the synapse that earned the reward

Complete distal(tau_e): each step, decay the traces by exp(-1/tau_e) and add 1 where a coincidence happened; deliver dopamine 100 to 200 steps after each coincidence at synapse 0; change the weights by eta (DA - DAbar) e and clip them to [0, 10].

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def distal(tau_e, eta=0.002, T=50000, seed=2007):
    r = np.random.default_rng(seed)
    N, delay = 100, 100
    w = np.full(N, 1.0); e = np.zeros(N); pending = []; DA = 0.0; DAbar = 0.1
    for t in range(T):
        pairs = r.random(N) < 0.005                  # random coincidences
        ...                                          # update the traces
        if pairs[0]:
            pending.append(t + delay + int(r.integers(0, 100)))
        DA *= np.exp(-1 / 20)
        while pending and pending[0] <= t:
            DA += 1; pending.pop(0)
        DAbar += (DA - DAbar) / 5000                 # slow running mean of dopamine
        w = ...
    return w

w_long, w_short = distal(200), distal(10)
print(f"trace 200 steps: synapse 0 {w_long[0]:.2f}, others mean {w_long[1:].mean():.2f}, max {w_long[1:].max():.2f}")
print(f"trace 10 steps:  synapse 0 {w_short[0]:.2f}, others mean {w_short[1:].mean():.2f}, max {w_short[1:].max():.2f}")
plt.bar(range(100), w_long); plt.xlabel("synapse"); plt.ylabel("final weight"); plt.show()

In [ ]:
import numpy as np
def test_found():
    "with a 200-step trace, synapse 0 ends up far above every other synapse"
    assert w_long[0] > 2 * w_long[1:].max()

def test_short():
    "with a 10-step trace, synapse 0 is not singled out"
    assert w_short[0] < w_short[1:].max()

test_found()
test_short()
print("All checks passed")

### Exercise: Reward-modulated Hebbian learning

Complete rm_hebb(baseline): on each trial draw x and exploration noise xi, compute y and the reward R = 20 - (y - w* x)², form the third factor M = R - Rbar (or R alone without a baseline), update w by eta M xi x and the running baseline Rbar by 0.05 (R - Rbar).

In [ ]:
r = np.random.default_rng(2016)
D = 10
w_star = r.normal(size=D)

def rm_hebb(baseline=True, trials=3000, eta=0.002, sig=0.5):
    w = np.zeros(D); Rbar = 0.0; errs = []
    for k in range(trials):
        x = r.normal(size=D); xi = sig * r.normal()
        y = w @ x + xi
        R = 20.0 - (y - w_star @ x) ** 2
        ...
        errs.append(np.sum((w - w_star) ** 2))
    return w, np.array(errs)

_, err_b = rm_hebb(True)
_, err_nb = rm_hebb(False)
print(f"squared weight error after learning: with baseline {np.mean(err_b[-200:]):.3f}, without {np.mean(err_nb[-200:]):.3f}")
plt.semilogy(err_b, label="R minus baseline"); plt.semilogy(err_nb, label="raw R"); plt.xlabel("trial"); plt.ylabel("|w - w*|²"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_learns():
    "with a baseline the weights converge close to the target"
    assert np.mean(err_b[-200:]) < 0.1

def test_baseline():
    "without a baseline the final error is at least ten times larger"
    assert np.mean(err_nb[-200:]) > 10 * np.mean(err_b[-200:])

test_learns()
test_baseline()
print("All checks passed")